# Chapter 13: Inheritance in Object-Oriented Programming

Inheritance is one of the four pillars of object-oriented programming (alongside **encapsulation**, **abstraction**, and **polymorphism**). It lets you define a new class that *inherits* the attributes and behaviors of an existing class — allowing you to reuse, extend, and specialize code without starting from scratch.

## Why Inheritance Matters

Imagine modeling a zoo. You have `Animal`, `Dog`, `Cat`, `Parrot`. Every animal has a name and can eat. But a `Dog` can also fetch, and a `Parrot` can talk. Without inheritance you'd copy the `name` and `eat()` logic into every class. With inheritance, you write it once in `Animal` and let the others *derive* from it.

## Chapter Road Map

| Section | Topic | Key Idea |
|---------|-------|----------|
| 13.1 | Derived classes | How child classes are created from parent classes |
| 13.2 | Accessing base class attributes | Using `super()` to reach the parent |
| 13.3 | Overriding class methods | Replacing parent behavior in the child |
| 13.4 | Is-a vs has-a | Two fundamental design relationships |
| 13.5 | Mixins & multiple inheritance | Composing behavior from many sources |
| 13.6 | `unittest` module | Testing your class hierarchy |
| 13.7–13.9 | Labs | Pets, Instruments, Courses |

---
## 13.1 — Derived Classes

### What Is a Derived Class?

A **derived class** (also called a *child* or *subclass*) is a class built on top of an existing **base class** (also called a *parent* or *superclass*). The child class automatically receives all of the parent's attributes and methods — this is the *inheritance* — and can then add new ones of its own.

### Syntax at a Glance

```python
class Parent:
    ...

class Child(Parent):   # <-- the parentheses declare the relationship
    ...
```

Python reads `Child(Parent)` as: *"Child is derived from Parent."*

### Key Points
- Every class in Python ultimately derives from the built-in `object` class.
- A derived class can be used anywhere the base class is expected (*substitutability*).
- You can chain inheritance as deep as you like, though shallow hierarchies are usually easier to reason about.

In [ ]:
# ── 13.1 Example: A simple class hierarchy ──────────────────────────────────

class Animal:
    """Base class for all animals."""

    def __init__(self, name: str, species: str):
        self.name = name
        self.species = species

    def eat(self):
        print(f"{self.name} is eating.")

    def describe(self):
        print(f"{self.name} is a {self.species}.")


class Dog(Animal):      # Dog derives from Animal
    """A dog — it does everything an Animal does, plus it can fetch."""

    def fetch(self, item: str):
        print(f"{self.name} fetches the {item}!")


class Parrot(Animal):   # Parrot also derives from Animal
    """A parrot — it does everything an Animal does, plus it can talk."""

    def talk(self, phrase: str):
        print(f"{self.name} says: '{phrase}'")


# ── Demonstration ────────────────────────────────────────────────────────────
rex    = Dog("Rex", "Canis lupus familiaris")
polly  = Parrot("Polly", "Amazona aestiva")

rex.describe()          # inherited from Animal
rex.eat()               # inherited from Animal
rex.fetch("tennis ball") # defined only on Dog

print()
polly.describe()        # inherited from Animal
polly.eat()             # inherited from Animal
polly.talk("Polly wants a cracker")  # defined only on Parrot

print()
# isinstance() respects the hierarchy
print(f"rex is an Animal: {isinstance(rex, Animal)}")
print(f"rex is a Dog:     {isinstance(rex, Dog)}")
print(f"rex is a Parrot:  {isinstance(rex, Parrot)}")

### Implementation Notes — 13.1

- **`class Dog(Animal)`** — the name in parentheses is the base class. Python looks up the inheritance chain automatically when resolving attributes.
- `Dog` and `Parrot` define **no `__init__`** of their own, so Python walks up the hierarchy and uses `Animal.__init__` automatically. This is the *inheritance of constructors* in action.
- **`isinstance(rex, Animal)`** returns `True` because `Dog` is a subclass of `Animal`. This is fundamental to polymorphism — you can treat a `Dog` wherever an `Animal` is expected.
- The method resolution order (MRO) can always be inspected: `Dog.__mro__`.

---
## 13.2 — Accessing Base Class Attributes

### The Problem

What if a child class needs its *own* `__init__` — perhaps to add extra attributes — but still needs to set up everything the parent defines? Rewriting the parent's initialization code would break the DRY principle and create a maintenance nightmare.

### The Solution: `super()`

`super()` returns a *proxy object* that delegates method calls to the parent class. It respects the Method Resolution Order (MRO), which is crucial for multiple inheritance.

```python
class Child(Parent):
    def __init__(self, parent_arg, child_arg):
        super().__init__(parent_arg)   # run Parent's __init__
        self.child_arg = child_arg     # then add our own stuff
```

### Key Points
- `super()` works for any method, not just `__init__`.
- Always call `super().__init__()` *before* adding your own attributes so the parent's state is ready.
- `super()` without arguments (Python 3 style) is the preferred form.

In [ ]:
# ── 13.2 Example: Extending the parent's __init__ with super() ───────────────

class Vehicle:
    """Base class for all vehicles."""

    def __init__(self, make: str, model: str, year: int):
        self.make  = make
        self.model = model
        self.year  = year

    def describe(self):
        print(f"{self.year} {self.make} {self.model}")


class ElectricVehicle(Vehicle):
    """An electric vehicle adds battery capacity to the base Vehicle."""

    def __init__(self, make: str, model: str, year: int, battery_kwh: float):
        super().__init__(make, model, year)  # delegate to Vehicle.__init__
        self.battery_kwh = battery_kwh       # then add our own attribute

    def describe(self):
        super().describe()                   # reuse parent's describe logic
        print(f"  Battery: {self.battery_kwh} kWh")


class HybridVehicle(Vehicle):
    """A hybrid adds both an engine size and a small battery."""

    def __init__(self, make, model, year, engine_l: float, battery_kwh: float):
        super().__init__(make, model, year)
        self.engine_l    = engine_l
        self.battery_kwh = battery_kwh

    def describe(self):
        super().describe()
        print(f"  Engine: {self.engine_l}L  |  Battery: {self.battery_kwh} kWh")


# ── Demonstration ────────────────────────────────────────────────────────────
tesla   = ElectricVehicle("Tesla", "Model 3", 2024, 82.0)
prius   = HybridVehicle("Toyota", "Prius", 2023, 1.8, 8.8)
classic = Vehicle("Ford", "Mustang", 1967)

for v in [tesla, prius, classic]:
    v.describe()
    print()

### Implementation Notes — 13.2

- **`super().__init__(make, model, year)`** — we pass all three arguments that `Vehicle.__init__` expects. The child doesn't duplicate that assignment logic; the parent handles it.
- **`super().describe()`** inside `ElectricVehicle.describe()` — we call the parent's version first, then *append* our own extra output. This is a clean pattern: extend, don't replace.
- If you forget `super().__init__()`, attributes like `self.make` will not exist on the child instance and any method that accesses them will raise an `AttributeError`.
- `super()` in Python 3 is smart — it knows which class and instance it belongs to without needing explicit arguments.

---
## 13.3 — Overriding Class Methods

### What Is Overriding?

A child class **overrides** a parent method by defining a method with the *same name*. When that method is called on an instance of the child, Python uses the child's version rather than the parent's. This is the mechanism behind **polymorphism** — the same method call can behave differently depending on the object's actual type.

### Patterns

| Pattern | Description |
|---------|-------------|
| **Full replacement** | Child defines the method completely; parent version is ignored |
| **Extension** | Child calls `super().method()` first, then adds to it |
| **Pre-processing** | Child does work first, then delegates to `super()` |

### Key Points
- Override is resolved at **runtime**, based on the actual type of the object — this is called *dynamic dispatch*.
- You can always still access the parent's original version explicitly via `super()`.
- Overriding is **not** the same as overloading (which Python does not natively support the way Java/C++ do).

In [ ]:
# ── 13.3 Example: Polymorphism via method overriding ────────────────────────

class Shape:
    """Base class. Every shape has a color and can report its area."""

    def __init__(self, color: str = "white"):
        self.color = color

    def area(self) -> float:
        """Subclasses must override this."""
        raise NotImplementedError("Subclasses must implement area()")

    def describe(self):
        print(f"{self.__class__.__name__} | color={self.color} | area={self.area():.2f}")


class Circle(Shape):
    import math

    def __init__(self, radius: float, color: str = "white"):
        super().__init__(color)
        self.radius = radius

    def area(self) -> float:           # ← overrides Shape.area
        import math
        return math.pi * self.radius ** 2


class Rectangle(Shape):

    def __init__(self, width: float, height: float, color: str = "white"):
        super().__init__(color)
        self.width  = width
        self.height = height

    def area(self) -> float:           # ← overrides Shape.area
        return self.width * self.height


class Square(Rectangle):
    """A square IS a rectangle with equal sides."""

    def __init__(self, side: float, color: str = "white"):
        super().__init__(side, side, color)  # reuse Rectangle fully

    # area() is inherited from Rectangle — no override needed!


# ── Polymorphism: same call, different behavior ──────────────────────────────
shapes = [
    Circle(5, "red"),
    Rectangle(4, 6, "blue"),
    Square(3, "green"),
]

for shape in shapes:
    shape.describe()   # Shape.describe() calls shape.area() — resolved dynamically

### Implementation Notes — 13.3

- **`raise NotImplementedError`** in `Shape.area()` — this is a design signal: *"I expect subclasses to provide this."* It turns a missing override into an explicit error rather than silent wrong behavior. (A more formal approach uses Python's `abc.ABC` and `@abstractmethod`.)
- **`self.__class__.__name__`** inside `Shape.describe()` — even though `describe` lives on the parent, `self.__class__` always reflects the *actual* type (`Circle`, `Rectangle`, etc.). This is dynamic dispatch at work.
- `Square` overrides nothing — it gets `area()` from `Rectangle` for free, because a square's area is still `width × height` (and both were set to `side` in `__init__`). Good inheritance means knowing when *not* to override.

---
## 13.4 — Is-a Versus Has-a Relationships

### The Two Fundamental Design Relationships

Before you decide whether to use inheritance, ask yourself a simple question:

> **"Is a [Child] a type of [Parent]?"** — If *yes*, use **inheritance** (*is-a*).
> 
> **"Does a [Class] have a [Component]?"** — If *yes*, use **composition** (*has-a*).

### Is-a (Inheritance)

- A `Dog` **is an** `Animal` → `class Dog(Animal)` ✅
- A `Savings Account` **is a** `Bank Account` → `class SavingsAccount(BankAccount)` ✅
- A `Car` **is a** `Engine`? ❌ — a Car *has* an Engine.

### Has-a (Composition)

- A `Car` **has an** `Engine` → `self.engine = Engine(...)` ✅
- A `Library` **has** many `Books` → `self.books = []` ✅

### Why Does It Matter?

Choosing the wrong relationship leads to fragile, confusing code. A classic mistake is inheriting just to reuse code even when the is-a test fails. *Favor composition over inheritance* when in doubt — it's generally more flexible.

In [ ]:
# ── 13.4 Example: Contrasting is-a and has-a ────────────────────────────────

# ── Has-a: Engine is a component of Car, not a parent ───────────────────────
class Engine:
    def __init__(self, horsepower: int, fuel_type: str):
        self.horsepower = horsepower
        self.fuel_type  = fuel_type

    def start(self):
        print(f"Engine ({self.horsepower}hp, {self.fuel_type}) roars to life.")


class GpsUnit:
    def navigate(self, destination: str):
        print(f"GPS: Navigating to {destination}.")


# ── Is-a: SportsCar IS a Car ─────────────────────────────────────────────────
class Car:
    """Car HAS an Engine and HAS a GpsUnit."""

    def __init__(self, make: str, model: str, engine: Engine):
        self.make   = make
        self.model  = model
        self.engine = engine           # has-a
        self.gps    = GpsUnit()        # has-a

    def start(self):
        print(f"{self.make} {self.model}: ")
        self.engine.start()            # delegate to the component

    def go_to(self, destination: str):
        self.gps.navigate(destination) # delegate to the component


class SportsCar(Car):                  # is-a Car
    """A SportsCar IS a Car with a turbo boost mode."""

    def __init__(self, make, model, engine, top_speed_mph: int):
        super().__init__(make, model, engine)
        self.top_speed_mph = top_speed_mph

    def boost(self):
        print(f"{self.make} {self.model} engages turbo! Top speed: {self.top_speed_mph} mph")


# ── Demonstration ────────────────────────────────────────────────────────────
v8 = Engine(450, "gasoline")
lambo = SportsCar("Lamborghini", "Huracán", v8, 202)

lambo.start()           # uses Car.start → delegates to Engine.start
lambo.go_to("Monaco")   # delegates to GpsUnit.navigate
lambo.boost()           # SportsCar-only method

print()
print("Relationship check:")
print(f"  lambo is a Car:       {isinstance(lambo, Car)}")
print(f"  lambo is an Engine:   {isinstance(lambo, Engine)}")

### Implementation Notes — 13.4

- `Car` stores `Engine` and `GpsUnit` as **instance attributes** — this is composition. The car *uses* them but is not one of them.
- `Car.start()` **delegates** to `self.engine.start()`. This is the power of composition: `Car` doesn't need to know *how* an engine starts, only that it *can*.
- `SportsCar` passes the `engine` all the way up to `Car.__init__` via `super()`. The engine is still a *component* of the car, not a parent class.
- `isinstance(lambo, Engine)` is `False` — a `SportsCar` is not an `Engine`. Inheritance would have made this confusingly true.
- **Rule of thumb:** If you can't pass the "is-a" test in plain English, reach for composition instead.

---
## 13.5 — Mixin Classes and Multiple Inheritance

### Multiple Inheritance

Python allows a class to inherit from **more than one parent**:

```python
class Child(ParentA, ParentB):
    ...
```

`Child` gets all attributes and methods from both parents. When the same method name exists in both, Python uses the **Method Resolution Order (MRO)** — a left-to-right, depth-first search that guarantees each class appears only once.

### Mixin Classes

A **mixin** is a class designed specifically to be mixed into other classes via multiple inheritance. Mixins:
- Provide a focused set of reusable methods
- Are **not** meant to be instantiated directly
- Don't define `__init__` (or keep it minimal)
- Follow the naming convention `SomethingMixin`

### Why Mixins?

They let you compose behavior horizontally — across class hierarchies — without forcing a rigid tree structure. Think of them like *plug-in capabilities*.

In [ ]:
# ── 13.5 Example: Mixins as plug-in capabilities ────────────────────────────

# ── Mixin 1: Anything that can be serialized to JSON ────────────────────────
import json

class JsonSerializableMixin:
    """Gives any class a to_json() method."""

    def to_json(self) -> str:
        return json.dumps(self.__dict__, indent=2)


# ── Mixin 2: Anything that can log its activity ──────────────────────────────
class LoggableMixin:
    """Gives any class a log() method."""

    def log(self, message: str):
        class_name = self.__class__.__name__
        print(f"[LOG] {class_name}: {message}")


# ── Mixin 3: Anything that can be compared by a key ─────────────────────────
class ComparableMixin:
    """Gives any class __lt__ based on a comparable_key() method."""

    def comparable_key(self):
        raise NotImplementedError

    def __lt__(self, other):
        return self.comparable_key() < other.comparable_key()

    def __eq__(self, other):
        return self.comparable_key() == other.comparable_key()


# ── Base class ───────────────────────────────────────────────────────────────
class Employee:
    def __init__(self, name: str, department: str, salary: float):
        self.name       = name
        self.department = department
        self.salary     = salary

    def describe(self):
        print(f"{self.name} ({self.department}) — ${self.salary:,.0f}/yr")


# ── Mix them all in ──────────────────────────────────────────────────────────
class SmartEmployee(JsonSerializableMixin, LoggableMixin, ComparableMixin, Employee):
    """An Employee that can serialize, log, and compare itself."""

    def __init__(self, name, department, salary):
        super().__init__(name, department, salary)

    def comparable_key(self):
        return self.salary   # sort by salary


# ── Demonstration ────────────────────────────────────────────────────────────
alice = SmartEmployee("Alice", "Engineering", 120_000)
bob   = SmartEmployee("Bob",   "Marketing",   95_000)
carol = SmartEmployee("Carol", "Engineering", 135_000)

alice.describe()
alice.log("Completed Q4 performance review.")
print(alice.to_json())

print("\nSorted by salary:")
for emp in sorted([alice, bob, carol]):
    emp.describe()

print("\nMRO for SmartEmployee:")
for cls in SmartEmployee.__mro__:
    print(f"  {cls}")

### Implementation Notes — 13.5

- **Mixin ordering matters.** In `class SmartEmployee(JsonSerializableMixin, LoggableMixin, ComparableMixin, Employee)`, Python resolves methods left to right. Put the most specific class (the real base, `Employee`) last.
- **`super()` works cooperatively** across the entire MRO chain. This is why you should always call `super().__init__()` even in mixins — it ensures the chain propagates correctly.
- **`__dict__`** in `JsonSerializableMixin.to_json()` returns all instance attributes as a dict. It's a quick serialization trick, though for production code a more explicit approach is safer.
- **`__mro__`** — print it to understand exactly which class Python will ask for any given attribute. It's a tuple, ordered from most-specific to least-specific, ending in `object`.
- The `ComparableMixin` pattern (implement `__lt__` and `__eq__` in terms of a `comparable_key`) mirrors Python's standard library `functools.total_ordering` decorator.

---
## 13.6 — Testing Your Code: The `unittest` Module

### Why Test?

Inheritance hierarchies can be tricky — a change to the base class affects every subclass. Automated tests let you catch regressions quickly and prove that your classes behave correctly.

### The `unittest` Framework

Python's built-in `unittest` module provides:

| Concept | Role |
|---------|------|
| `TestCase` | A class you subclass to group related tests |
| `test_*` methods | Any method starting with `test_` is run as a test |
| `setUp()` | Runs before *each* test method — create fresh objects here |
| `tearDown()` | Runs after each test — clean up resources |
| `assert*` methods | `assertEqual`, `assertIsInstance`, `assertRaises`, etc. |

### Key Points
- Each test should be **independent** — don't rely on state left by another test.
- Test the *contract* of a class: what it promises to do, not how it does it internally.
- In a Jupyter notebook, call `unittest.main(argv=[''], exit=False)` to run tests inline.

In [ ]:
# ── 13.6 Example: Unit-testing an inheritance hierarchy ─────────────────────
import unittest

# ── Classes under test ───────────────────────────────────────────────────────
class BankAccount:
    def __init__(self, owner: str, balance: float = 0.0):
        self.owner   = owner
        self.balance = balance

    def deposit(self, amount: float):
        if amount <= 0:
            raise ValueError("Deposit must be positive.")
        self.balance += amount

    def withdraw(self, amount: float):
        if amount > self.balance:
            raise ValueError("Insufficient funds.")
        self.balance -= amount


class SavingsAccount(BankAccount):
    INTEREST_RATE = 0.03

    def apply_interest(self):
        self.balance += self.balance * self.INTEREST_RATE


# ── Tests ────────────────────────────────────────────────────────────────────
class TestBankAccount(unittest.TestCase):

    def setUp(self):
        """Fresh account before every test."""
        self.account = BankAccount("Alice", 100.0)

    def test_initial_balance(self):
        self.assertEqual(self.account.balance, 100.0)

    def test_deposit_increases_balance(self):
        self.account.deposit(50)
        self.assertEqual(self.account.balance, 150.0)

    def test_deposit_negative_raises(self):
        with self.assertRaises(ValueError):
            self.account.deposit(-10)

    def test_withdraw_decreases_balance(self):
        self.account.withdraw(40)
        self.assertEqual(self.account.balance, 60.0)

    def test_overdraft_raises(self):
        with self.assertRaises(ValueError):
            self.account.withdraw(200)


class TestSavingsAccount(unittest.TestCase):

    def setUp(self):
        self.savings = SavingsAccount("Bob", 1000.0)

    def test_is_bank_account(self):
        """Savings IS-A BankAccount."""
        self.assertIsInstance(self.savings, BankAccount)

    def test_interest_applied_correctly(self):
        self.savings.apply_interest()
        self.assertAlmostEqual(self.savings.balance, 1030.0, places=2)

    def test_inherited_deposit(self):
        """Inherited methods work on the child."""
        self.savings.deposit(500)
        self.assertEqual(self.savings.balance, 1500.0)


# ── Run inside Jupyter ────────────────────────────────────────────────────────
unittest.main(argv=[""], exit=False, verbosity=2)

### Implementation Notes — 13.6

- **`setUp()`** creates a fresh `BankAccount` before each test. This means `test_deposit` and `test_withdraw` never interfere with each other — they each start from `balance=100.0`.
- **`assertRaises` as a context manager** (`with self.assertRaises(ValueError):`) is the cleanest way to verify that a block of code raises an exception. The test passes only if `ValueError` is raised.
- **`assertAlmostEqual`** compares floats to a given number of decimal places — essential for financial or scientific calculations where floating-point precision matters.
- **`test_is_bank_account`** in `TestSavingsAccount` validates the inheritance relationship itself using `assertIsInstance`. This is good practice in class hierarchies.
- **`verbosity=2`** in `unittest.main()` prints each test name and result, making it easy to see exactly what passed or failed.

---
## 13.7 — LAB: Pet Information (Derived Classes)

### Objective

Build a `Pet` base class and derive specific pet types from it. Each pet type has shared attributes (name, age) but also unique behaviors.

### Design

```
Pet
├── Dog
└── Cat
```

- `Pet` — name, age, `speak()`, `info()`
- `Dog(Pet)` — adds breed, overrides `speak()`, adds `fetch()`
- `Cat(Pet)` — adds indoor/outdoor preference, overrides `speak()`, adds `purr()`

In [ ]:
# ── 13.7 LAB: Pet Information ────────────────────────────────────────────────

class Pet:
    """Base class for all pets."""

    def __init__(self, name: str, age: int):
        self.name = name
        self.age  = age

    def speak(self) -> str:
        return "..."

    def info(self):
        print(f"Name : {self.name}")
        print(f"Age  : {self.age} year(s)")
        print(f"Sound: {self.speak()}")


class Dog(Pet):
    """A dog with a known breed."""

    def __init__(self, name: str, age: int, breed: str):
        super().__init__(name, age)
        self.breed = breed

    def speak(self) -> str:            # override
        return "Woof!"

    def fetch(self, item: str):
        print(f"{self.name} the {self.breed} fetches the {item}!")

    def info(self):
        super().info()                 # extend
        print(f"Breed: {self.breed}")


class Cat(Pet):
    """A cat that may prefer indoors or outdoors."""

    def __init__(self, name: str, age: int, indoor: bool = True):
        super().__init__(name, age)
        self.indoor = indoor

    def speak(self) -> str:            # override
        return "Meow!"

    def purr(self):
        print(f"{self.name} purrs contentedly...")

    def info(self):
        super().info()
        preference = "Indoor" if self.indoor else "Outdoor"
        print(f"Type : {preference} cat")


# ── Demonstration ─────────────────────────────────────────────────────────────
pets = [
    Dog("Rex", 3, "German Shepherd"),
    Cat("Whiskers", 5, indoor=True),
    Dog("Buddy", 1, "Golden Retriever"),
    Cat("Shadow", 7, indoor=False),
]

for pet in pets:
    print("=" * 30)
    pet.info()
    if isinstance(pet, Dog):
        pet.fetch("stick")
    elif isinstance(pet, Cat):
        pet.purr()
print("=" * 30)

### Lab Notes — 13.7

- `info()` in both `Dog` and `Cat` calls `super().info()` first — this ensures the shared fields (name, age, sound) always print, and the subclass just appends its own extra line.
- The loop iterates over a mixed list of `Dog` and `Cat` objects. Calling `pet.info()` and `pet.speak()` works for all of them because they're all `Pet` instances — this is **polymorphism in practice**.
- `isinstance` branching inside a loop is occasionally necessary (as shown), but in general, prefer putting type-specific behavior inside the class itself via overriding rather than checking types externally.

---
## 13.8 — LAB: Instrument Information (Derived Classes)

### Objective

Model musical instruments using a base `Instrument` class and derived types organized by instrument family.

### Design

```
Instrument
├── StringInstrument
│   ├── Guitar
│   └── Violin
└── WindInstrument
    ├── Trumpet
    └── Flute
```

In [ ]:
# ── 13.8 LAB: Instrument Information ────────────────────────────────────────

class Instrument:
    """Base class for all musical instruments."""

    def __init__(self, name: str, origin_year: int):
        self.name        = name
        self.origin_year = origin_year

    def play(self) -> str:
        return "♩ ♪ ♫"

    def info(self):
        print(f"Instrument   : {self.name}")
        print(f"Origin ~year : {self.origin_year}")
        print(f"Sound        : {self.play()}")


# ── Mid-level: Instrument families ───────────────────────────────────────────
class StringInstrument(Instrument):
    """All string instruments share a number-of-strings attribute."""

    def __init__(self, name: str, origin_year: int, num_strings: int):
        super().__init__(name, origin_year)
        self.num_strings = num_strings

    def info(self):
        super().info()
        print(f"Strings      : {self.num_strings}")


class WindInstrument(Instrument):
    """All wind instruments track whether they're woodwind or brass."""

    def __init__(self, name: str, origin_year: int, family: str):
        super().__init__(name, origin_year)
        self.family = family

    def info(self):
        super().info()
        print(f"Family       : {self.family}")


# ── Leaf classes: specific instruments ───────────────────────────────────────
class Guitar(StringInstrument):
    def __init__(self, style: str = "acoustic"):
        super().__init__("Guitar", 1779, 6)
        self.style = style

    def play(self) -> str:
        return "🎸 Strum strum strum"

    def info(self):
        super().info()
        print(f"Style        : {self.style}")


class Violin(StringInstrument):
    def __init__(self):
        super().__init__("Violin", 1555, 4)

    def play(self) -> str:
        return "🎻 Serenade..."


class Trumpet(WindInstrument):
    def __init__(self):
        super().__init__("Trumpet", 1820, "Brass")

    def play(self) -> str:
        return "🎺 Toot toot!"


class Flute(WindInstrument):
    def __init__(self):
        super().__init__("Flute", 1670, "Woodwind")

    def play(self) -> str:
        return "🪈 Flit flit flit"


# ── Demonstration ─────────────────────────────────────────────────────────────
orchestra = [Guitar("electric"), Violin(), Trumpet(), Flute()]

for instrument in orchestra:
    print("─" * 35)
    instrument.info()
print("─" * 35)

print("\nAll playing together:")
print("  ", "  ".join(i.play() for i in orchestra))

### Lab Notes — 13.8

- This lab introduces a **three-level hierarchy**: `Instrument → StringInstrument → Guitar`. Each level adds detail without repeating the levels above it.
- `Guitar.info()` calls `super().info()` which calls `StringInstrument.info()` which calls `Instrument.info()`. The chain is fully cooperative — Python's MRO ensures each level runs exactly once in the right order.
- Leaf classes like `Guitar` and `Violin` set their own `origin_year` and `num_strings` in their `__init__` — the user of `Guitar()` doesn't need to know those implementation details.
- The `play()` override at each leaf class demonstrates how the same method call gives instrument-specific output when used polymorphically in the `orchestra` loop.

---
## 13.9 — LAB: Course Information (Derived Classes)

### Objective

Model a university course catalog using inheritance. A base `Course` covers common fields; derived types handle the differences between in-person, online, and lab courses.

### Design

```
Course
├── InPersonCourse
├── OnlineCourse
└── LabCourse
```

In [ ]:
# ── 13.9 LAB: Course Information ─────────────────────────────────────────────

class Course:
    """Base class representing any university course."""

    def __init__(self, title: str, code: str, credits: int, instructor: str):
        self.title      = title
        self.code       = code
        self.credits    = credits
        self.instructor = instructor
        self._students: list[str] = []

    def enroll(self, student_name: str):
        self._students.append(student_name)
        print(f"{student_name} enrolled in {self.code}.")

    def roster(self):
        if not self._students:
            print(f"{self.code}: No students enrolled.")
            return
        print(f"{self.code} Roster:")
        for i, name in enumerate(self._students, 1):
            print(f"  {i}. {name}")

    def info(self):
        print(f"Course     : {self.title} ({self.code})")
        print(f"Instructor : {self.instructor}")
        print(f"Credits    : {self.credits}")


class InPersonCourse(Course):
    """A traditional classroom course with a fixed room and schedule."""

    def __init__(self, title, code, credits, instructor,
                 room: str, schedule: str, capacity: int):
        super().__init__(title, code, credits, instructor)
        self.room     = room
        self.schedule = schedule
        self.capacity = capacity

    def enroll(self, student_name: str):
        if len(self._students) >= self.capacity:
            print(f"Cannot enroll {student_name}: {self.code} is at capacity ({self.capacity}).")
            return
        super().enroll(student_name)

    def info(self):
        super().info()
        print(f"Room       : {self.room}")
        print(f"Schedule   : {self.schedule}")
        print(f"Capacity   : {self.capacity}")


class OnlineCourse(Course):
    """A fully asynchronous online course with a URL."""

    def __init__(self, title, code, credits, instructor, url: str, self_paced: bool = True):
        super().__init__(title, code, credits, instructor)
        self.url        = url
        self.self_paced = self_paced

    def info(self):
        super().info()
        pacing = "Self-paced" if self.self_paced else "Instructor-paced"
        print(f"Format     : Online — {pacing}")
        print(f"URL        : {self.url}")


class LabCourse(InPersonCourse):
    """A lab course — inherits in-person constraints and adds equipment needs."""

    def __init__(self, title, code, credits, instructor,
                 room, schedule, capacity, equipment: list[str]):
        super().__init__(title, code, credits, instructor, room, schedule, capacity)
        self.equipment = equipment

    def info(self):
        super().info()
        print(f"Equipment  : {', '.join(self.equipment)}")


# ── Demonstration ─────────────────────────────────────────────────────────────
cs101 = InPersonCourse(
    "Intro to CS", "CS101", 3, "Dr. Smith",
    room="Hall 204", schedule="MWF 9–10am", capacity=3
)

py200 = OnlineCourse(
    "Python for Everyone", "PY200", 3, "Prof. Lee",
    url="https://learn.university.edu/py200", self_paced=True
)

bio_lab = LabCourse(
    "Cell Biology Lab", "BIO310", 2, "Dr. Nguyen",
    room="Science Bldg B", schedule="Tue 1–4pm", capacity=20,
    equipment=["Microscope", "Centrifuge", "Pipettes"]
)

print("=" * 40)
cs101.info()
print()
for student in ["Alice", "Bob", "Carol", "Dave"]:  # Dave will be turned away
    cs101.enroll(student)
print()
cs101.roster()

print("\n" + "=" * 40)
py200.info()

print("\n" + "=" * 40)
bio_lab.info()

### Lab Notes — 13.9

- **`InPersonCourse.enroll()`** overrides `Course.enroll()` to add a capacity check *before* delegating to `super().enroll()`. This is the *pre-processing* override pattern — the child does validation, then lets the parent do the actual work.
- **`LabCourse`** extends `InPersonCourse` (not `Course` directly) — it inherits the capacity-guarded `enroll()` for free. This is a three-level chain: `LabCourse → InPersonCourse → Course`.
- **`_students`** uses a leading underscore to signal it's an *internal* attribute — callers should use `enroll()` and `roster()` rather than manipulating the list directly. This is encapsulation working alongside inheritance.
- Notice that `info()` at each level calls `super().info()` — the printed output is built up layer by layer, each subclass appending its own fields. This cooperative pattern scales cleanly as you add more subclasses.

---
## Chapter 13 Summary

| Concept | One-Line Takeaway |
|---------|-------------------|
| Derived classes | `class Child(Parent)` — automatic attribute/method inheritance |
| `super()` | The clean way to reach up the chain without hardcoding class names |
| Overriding | Same method name → child's version wins; use `super()` to extend |
| Is-a vs has-a | Inheritance for types; composition for components |
| Mixins | Plug-in behaviors via multiple inheritance; keeps classes focused |
| `unittest` | Each test is independent; assert the contract, not the implementation |

> *"Inheritance is not about code reuse — it's about expressing a true type relationship. When in doubt, compose."*